# Part III: Deciphering Function and Safety from the Genome – The Core of Probiotic Evaluation

With a high‑quality, annotated genome in hand, we now turn to the central question: **does this bacterium possess the genetic potential to be a safe and effective probiotic?** Answering this requires a systematic mining of the genome for metabolic pathways, beneficial traits, and safety liabilities. This part builds directly on the annotations produced in Part II. Each chapter focuses on a specific layer of functional evidence, always linking genotype to phenotype and emphasizing that computational predictions are hypotheses that must be weighed against one another.

## Chapter 5: Metabolic Potential – Reconstructing Pathways from KEGG and COG

A probiotic’s ability to produce beneficial metabolites (short‑chain fatty acids, vitamins) and to utilize prebiotic substrates depends on the presence of complete metabolic pathways. The starting point is the set of protein sequences predicted by Prokka. We will assign them to functional categories and then map them onto biochemical pathways.

### 5.1 Protein Orthology Assignment with eggNOG‑mapper

While Prokka already provides gene names and EC numbers, a more systematic functional overview is obtained by assigning each protein to a **Cluster of Orthologous Groups (COG)** or **eggNOG** category. We use **eggNOG‑mapper**, which employs precomputed hidden Markov models and Diamond searches. The statistical significance of a match is measured by an E‑value derived from the Karlin–Altschul equation, which for a given similarity score $S$ is:

$$
E = K m n e^{-\lambda S}
$$

Here $m$ and $n$ are the lengths of the query and database, $K$ and $\lambda$ are normalisation parameters. A low E‑value indicates a biologically meaningful assignment.

**Command:**
```bash
emapper.py -i proteins.faa --output results/annotation/eggnog \
           --cpu 12 \
           --itype proteins --tax_scope auto
```

**Key output files:**
- `eggnog.emapper.annotations`: Tab‑separated file with columns for query gene, eggNOG orthogroup, preferred name, COG category, KEGG KO, EC number, and GO terms.
- `eggnog.emapper.hmm_hits`: Detailed alignment metrics.

**Interpreting COG categories:**
COG categories are single‑letter codes that group proteins into broad functional classes. For a probiotic, we are particularly interested in:

| COG code | Functional category | Probiotic relevance |
|----------|---------------------|---------------------|
| G | Carbohydrate transport and metabolism | Prebiotic utilization |
| E | Amino acid transport and metabolism | Nitrogen source utilization, bioactive peptide production |
| H | Coenzyme transport and metabolism | Vitamin biosynthesis (e.g., folate, riboflavin) |
| I | Lipid transport and metabolism | Short‑chain fatty acid (SCFA) production |
| P | Inorganic ion transport and metabolism | Mineral acquisition (iron, zinc), stress response |
| V | Defense mechanisms | Bacteriocin production, restriction‑modification |
| M | Cell wall/membrane/envelope biogenesis | Surface protein anchoring, adhesion |

By tabulating the COG distribution of your isolate and comparing it to known probiotic strains, you obtain a first functional fingerprint.

### 5.2 KEGG Pathway Reconstruction

**KEGG (Kyoto Encyclopedia of Genes and Genomes)** provides a hierarchical organization of biological pathways. The critical identifier is the **KEGG Orthology (KO)** number, which uniquely defines a functional role in a pathway. eggNOG‑mapper already assigned KO numbers to many of your proteins; you can also use **BlastKOALA** (web, https://www.kegg.jp/blastkoala/) or the standalone **kofamscan** to obtain refined KO annotations.

**Running kofamscan:**
```bash
exec_annotation proteins.faa -o results/annotation/kofamscan.txt \
                 --profile /path/to/kofam/profiles/ \
                 --ko-list /path/to/kofam/ko_list
```

**Pathway mapping:**
Once KO numbers are assigned, reconstruct pathway completeness using tools such as:
- **KEGG Mapper – Reconstruct Pathway** (web): Upload your KO list to see which enzymes are present in each pathway module. The tool paints enzymes green (present) on reference maps.
- **MicroCyc** (Pathway Tools): If you have a complete genome, you can generate a pathway‑genome database.
- **MinPath**: A parsimony‑based approach that estimates the minimal set of pathways supported by your gene list, avoiding the pitfall of inferring complex pathways from a few hits.

**Focus pathways for probiotic evaluation:**
1. **Short‑chain fatty acid biosynthesis**: Acetate (from pyruvate via acetyl‑CoA), propionate (succinate or propanediol pathway), and butyrate (acetyl‑CoA → butyryl‑CoA). Key enzymes include phosphate acetyltransferase (K00625), acetate kinase (K00925), butyrate kinase (K00929), butyryl‑CoA dehydrogenase (K00248). Check whether the pathway is complete and not truncated.
2. **Vitamin biosynthesis**: Folate (KEGG module M00126), riboflavin (M00125), cobalamin (M00122). Probiotic strains that can synthesize vitamins may offer nutritional benefits.
3. **GABA (γ‑aminobutyric acid) production**: Glutamate decarboxylase (gadB/gadA, K01580) converts glutamate to GABA, a neurotransmitter with reported health effects. The presence of the *gad* gene cluster and a glutamate/GABA antiporter suggests GABA production capacity.
4. **Bile salt metabolism**: Bile salt hydrolase (BSH, K01442) catalyzes deconjugation of bile acids, which can influence cholesterol metabolism and gut colonization. Although not a pathway per se, the enzyme is easily identified and should be noted.

**Visualization and reporting:**
Generate bar charts showing the proportion of genes assigned to each KEGG category (Metabolism, Genetic Information Processing, Environmental Information Processing, etc.). Highlight complete modules relevant to probiotic function, as incomplete modules may indicate a pseudogenized or non‑functional pathway.

### 5.3 Using BlastKOALA for Rapid Profiling

For a quick assessment without local command‑line setup, use the BlastKOALA web service at https://www.kegg.jp/blastkoala/. Upload your protein FASTA, set the taxonomy to Bacteria, and after analysis download the annotation and pathway reconstruction files. The “Module reconstruction” output explicitly lists complete, incomplete, and missing modules—a convenient starting point.


### Case study 3: Pathway reconstruction

Pathway reconstruction is essential for understanding the biological functions that a newly sequenced genome encodes. Let's try with BlastKOALA for, *Intestinimonas butyriciproducens* and *L.plantarum*:
- Assign KEGG orthology to your genes predicted by prokka
- Reconstruct your KEGG pathway with the KEGG orthology list
- Record all necessary information (KEGG pathway present in the genome, KEGG module completeness)


## Chapter 6: Mining for Probiotic‑Associated Traits

Beyond core metabolic pathways, candidate probiotics must exhibit specific functional traits: the ability to break down complex dietary carbohydrates, to produce antimicrobial substances, and to adhere to and interact with the host intestinal epithelium. These traits are encoded by distinct gene clusters that can be systematically detected.

### 6.1 Carbohydrate‑Active Enzymes (CAZymes) and Prebiotic Utilization

The CAZy database (http://www.cazy.org/) classifies enzymes that assemble or break down glycosidic bonds. The presence of diverse glycoside hydrolases (GHs) and polysaccharide lyases (PLs) indicates the capacity to utilize a wide range of dietary fibers and host glycans.

**Tool: run_dbcan**
`run_dbcan` is a standalone pipeline of dbCAN3(https://pro.unl.edu/dbCAN2/). Check the official [document](https://run-dbcan.readthedocs.io/en/latest/) and [github](https://github.com/bcb-unl/run_dbcan)

```bash
run_dbcan.py proteins.faa protein \
            --out_dir results/cazyme \
            --db_dir /path/to/db \
            --tools hmmer diamond hotpep \
            --cpu 12
```

**Output interpretation:**
- Focus on families known to degrade plant cell wall components: GH5, GH9, GH10, GH11, PL1, PL9.
- Fucosidases (GH29, GH95) and sialidases (GH33) suggest the ability to forage host mucosal glycans—a trait associated with persistent colonizers. Check [Ref](https://www.nature.com/articles/s41467-023-37533-6)
- PULs (polysaccharide utilization loci): In *Bacteroides* and related genera, CAZymes are often organized in large clusters with SusC/SusD transporters. Database like [Polysaccharide-Utilization Loci DataBase](https://www.cazy.org/PULDB/) or manual inspection of genomic neighborhoods around GH genes can identify these loci, which are strong indicators of prebiotic utilization.

```{Tips}

Check CAZypedia: https://www.cazypedia.org/index.php/CAZypedia:About

```


### Case study 4: Carbohydrate utilization ability

Carbohydrate utilization is your body's ability to digest, convert, and use sugars and starches for fuel. Your body relies heavily on carbohydrates as its primary and preferred source of energy for the brain and muscles. Effective processing helps sustain steady energy and prevents blood sugar crashes. Therefore, it is important to recognize the ability of the potential probiotics, *Intestinimonas butyriciproducens* and the *L.plantarum*, in utlizing the complex carbohyrate.
- Use dbCAN3 web server to run with the genome and protein fasta file.
- Draw a plot to compare two strains.

### 6.2 Secondary Metabolite Biosynthetic Gene Clusters (antiSMASH/BAGEL4)

Biosynthetic Gene Clusters (BGCs) are physically grouped sets of genes within an organism's DNA that work together to produce a specific specialized (or secondary) metabolite. 

**antiSMASH** (antibiotics & Secondary Metabolite Analysis Shell) detects and annotates biosynthetic gene clusters (BGCs) for polyketides, non‑ribosomal peptides, bacteriocins, terpenes, and more. Probiotic strains often use bacteriocins to inhibit competitors. [BAGEL4](http://bagel4.molgenrug.nl/) server can also be used for these purpose.


```bash

antismash --genefinding-tool prodigal \
          --output-dir results/antismash \
          --cpus 12 \
          assembly.fasta

```

**Key BGC types for probiotics:**
- **Bacteriocins**: Lanthipeptides (class I), thiopeptides, sactipeptides. Lantibiotics such as nisin (class I) or lacticin 3147 are well‑characterized antimicrobials. A complete lantibiotic cluster includes the structural peptide, modification enzymes (LanB, LanC, LanM), a transporter, and an immunity protein.
- **RiPPs** (Ribosomally synthesized and Post‑translationally modified Peptides): Class II bacteriocins (pediocin‑like) and other unmodified peptides.
- **Terpenes**: Although not directly antimicrobial, some terpenes have immunomodulatory properties.

**Validation:**  
Not every predicted cluster is functional. Examine the completeness of the core biosynthetic machinery, the presence of a dedicated transporter, and the similarity to known clusters with characterized products (antiSMASH provides a `knownclusterblast` comparison). A cluster that shares >80% of genes with a known bacteriocin cluster is highly likely to produce a similar compound.

### Case study 5: Biosynthetic Gene Clusters
Screen the Bacteriocins and RiPPs using BAGEL4 server
- Write description.


### 6.3 Trivial probiotics-associated genes

In the section 6.3, we will introduce several probiotics-associated genes.

```{Note}

Please intepret very carefully for the results derived from this session.

```

#### 6.3.1 Host Interaction and Stress Tolerance Genes

Probiotics must survive stomach acid, bile, and osmotic stress, and then adhere to intestinal mucus or epithelial cells. We search the proteome for established markers.


##### Adhesion Factors

- **Surface layer proteins (S‑layers)**: Often identified by BLAST against the Pfam domains, or by scanning for large (100–200 kDa) proteins with a signal peptide and a C‑terminal cell‑wall‑anchoring motif (LPxTG for Gram‑positives). An S‑layer gene suggests the strain forms a crystalline surface array that can mediate adhesion.
- **Mucus‑binding proteins (MUBs)** and **adhesins**: Domains such as MucBP (PF06458), *Lactobacillus* species often encode SpaC or SpaA pilus subunits that mediate binding to mucin.
- **Fibronectin‑binding proteins**: Recognized by Pfam domains PF02986, PF05833.
- **Biofilm‑associated genes**: e.g., *ica* operon in staphylococci, or *bscA/bscB* in *Lactobacillus*. Exopolysaccharide (EPS) biosynthesis clusters can be identified with antiSMASH or by searching for genes similar to the *eps* operon glycosyltransferases.

##### Stress Resistance Mechanisms

- **Acid tolerance**: The glutamate decarboxylase (GAD) system (*gadB/gadC*) and the arginine deiminase (ADI) pathway (*arcA/arcB/arcC*) consume protons during amino acid decarboxylation, buffering the cytoplasm. Also look for the F₁F₀‑ATPase operon (proton extrusion) and chaperones (GroEL, DnaK).
- **Bile resistance**: Bile salt hydrolase (BSH, K01442) and multidrug efflux pumps (MepA, MdtK) are common. The BSH gene can be identified by its conserved domain and active‑site motif.
- **Oxidative and osmotic stress**: Catalase (*kat*), superoxide dismutase (*sod*), and the uptake systems for compatible solutes (glycine betaine, proline) encoded by *proU*, *opuA*, etc.

#### 6.3.2 Vitamins and Health‑Promoting Metabolites

Beyond the KEGG pathway level, confirm the presence of complete gene sets for vitamin synthesis:
- **Folate (vitamin B9)**: *fol* genes (folA, folC, folE, folP, folK, folB, folQ). The absence of any *fol* gene may result in auxotrophy.
- **Riboflavin (vitamin B2)**: *rib* operon (ribA through ribH). Some strains produce riboflavin as a by‑product, improving nutritional value.
- **Exopolysaccharides (EPS)**: EPS clusters (identified via antiSMASH or manual inspection of glycosyltransferase‑rich regions) contribute to texture, immunomodulation, and biofilm formation.

A quick script can search for keywords in the Prokka annotation and flag the presence/absence of these traits, which you can then verify by examining gene context.


### Case study 6: Probiotics-associated gene

This part is typically messy and lack of clear standard. You may have a look at the probiotics associated genes list as described in the article here (https://doi.org/10.1080/17460913.2025.2492472). Then through a key words searching either by R (suggested) or by hand. Keep in mind that, the presence of certain genes may be only **associated** with certain functions.
- Screen the probiotics associated gene.
- Write description 


Use gen-AI to help you generate the scripts for key words searching!

- **Input**: list of gene name, and the annotation file such as eggnog mapper
- **Output**: whether the gene was present in the annotation file

## Chapter 7: Safety Assessment – Virulence, Antibiotic Resistance, and Genomic Integrity

A bacterium cannot be considered a probiotic until it is shown to be **safe for human consumption**. Genomic safety screening is the first, indispensable step. For food industry, even the most promising metabolic profile is USELESS if the genome harbors transferable antibiotic resistance genes or known virulence factors.

```{Note}

In this chapter, galaxy (https://usegalaxy.eu/) and proksee platform will be used again.

```


### 7.1 Detection of Virulence Factors

**Database: VFDB (Virulence Factors Database)**  
The VFDB contains curated sets of experimentally verified virulence factors from pathogenic bacteria. Using BLAST, we compare our strain’s proteins against the VFDB reference set.

**Tool: ABRicate**  
ABRicate bundles multiple databases (VFDB, CARD, PlasmidFinder, etc.) and automates the screening with a consistent interface. abricate can be run through galaxy.

```bash

# Run VFDB screening
abricate --db vfdb results/annotation/sample.gbk > results/safety/vfdb_results.tab

# Summarize per sample
abricate --summary results/safety/vfdb_results.tab > results/safety/vfdb_summary.tab
```

**Interpretation:**
- **High‑risk genes**: Those encoding toxins (e.g., hemolysin, enterotoxin, botulinum toxin), invasins, type III secretion system components, and key regulators of virulence. A match with >70% identity and >80% coverage to a *known* toxic protein is a red flag.
- **Context matters**: Some genes can be both virulence factors and housekeeping components in non‑pathogenic species (e.g., certain adhesion factors). Determine whether the matched gene is part of a larger virulence operon, its genomic neighborhood (plasmid? phage?), and its phylogenetic distribution. A gene present in all strains of a generally recognized as safe (GRAS) species is less concerning than a rare, plasmid‑borne toxin.
- **Reporting**: Categorize hits as “matches requiring investigation” and “matches consistent with commensal/GRAS lifestyle.”

### 7.2 Antibiotic Resistance Gene Screening

**Database: CARD (Comprehensive Antibiotic Resistance Database)** and **ResFinder**  
These databases catalog antibiotic resistance determinants: acquired genes, mutations that confer resistance, and intrinsic resistance alleles. The table below contrasts their main features.

| Feature | CARD | ResFinder |
|---------|------|-----------|
| Scope | All resistance determinants (genes, SNPs) | Acquired antimicrobial resistance genes |
| Evidence | Expert curated, ontology‑based | Curated from literature, minimal SNPs |
| Output | ARO terms, AMR gene families | Gene names, % identity, phenotype |
| Best for | Comprehensive resistance profiling, mechanism prediction | Rapid, targeted screening for acquired resistance genes |
| Use in regulation | Widely cited in research | Endorsed by EFSA for food/feed safety |

**ABRicate with CARD and ResFinder:**
```bash
abricate --db card results/annotation/sample.gbk > results/safety/card_results.tab
abricate --db resfinder results/annotation/sample.gbk > results/safety/resfinder_results.tab
```

**Critical analysis questions:**
- **Is the resistance gene complete?** A truncated gene with missing essential domains is likely non‑functional. Check the alignment coverage and the presence of the full reading frame.
- **Is it a mobile genetic element?** If the resistance gene is located on a plasmid, within an integron, or adjacent to transposase/integrase genes, the risk of horizontal transfer is high. Tools like **PlasmidFinder** (via ABRicate) and **mobileOG‑db** can help identify mobile context.
- **Intrinsic vs. acquired**: Many species have intrinsic, non‑transferable resistance (e.g., *Lactobacillus* to vancomycin). Annotate such cases explicitly and support with literature.
- **“Selection for resistance” risk**: Even if a resistance gene is chromosomal and seemingly stable, regulatory bodies may still reject its presence in a commercial probiotic. The European Food Safety Authority (EFSA) provides guidance: strains intended for food/feed chains must not harbor acquired antimicrobial resistance genes.

**Reporting:**  
Present a table of all detected resistance genes with columns: Gene, product, %identity, %coverage, location (chromosome/plasmid), mobile element association, and an assessment of risk (low/medium/high).


### Case study 7: Integrating the Safety Profile

At the end of this chapter, you should compile a **safety risk assessment** for the strain. Let's focus on the most important parts.

1. **Virulence factors**: List all matches, their significance, and a final risk level (absent, low‑concern, high‑concern).
2. **Antibiotic resistance**: Describe each resistance gene, its genomic context, and transferability potential.
5. **Overall safety verdict**: “Likely safe, subject to phenotypic confirmation” or “Unsafe due to presence of X, Y, Z.”

This structured assessment, combined with the functional potential mapped in Chapters 5 and 6, forms the genomic core of your probiotic evaluation report.


# Open Question

As we are investigating the probiotics for ALD, can we find direct association between bacterial gene and ethanol injury?
